<a href="https://colab.research.google.com/github/runessaa/-Streltsov-Projects/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%9612.1_%D0%A0%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%D1%81_%D0%B2%D0%B5%D0%BA%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%BC%D0%B8_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%BC%D0%B8_%D0%B2_GeoPandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №4. Работа с векторными данными в GeoPandas**

Выполняя практическую работу, опирайтесь на теоритический материал, рассматриваемый на занятии:

- Использование NumPy, Pandas и GeoPandas для работы с пространственными данными: https://u.to/rk86Ig
- Ультимативный обзор аналитических инструментов GeoPandas:  https://u.to/s0d3Ig
- Теоретические аспекты и практические примеры получения векторных данных из OpenStreetMap: https://u.to/9SdwIQ

Дополнительно:

- Особенности построения интерактивных карт с помощью библиотеки Leafmap: https://u.to/WvA3Ig

Официальная техническая документацию по используемым библиотекам:

- **NumPy**: https://numpy.org/doc/stable/
- **Pandas**: https://pandas.pydata.org/docs/
- **GeoPandas**: https://geopandas.org/en/stable/docs.html
- **Leafmap**: https://leafmap.org/

In [1]:
%%capture
!pip install geopandas leafmap mapclassify # Устанавливаем библиотеку GeoPandas и необходимые зависимости

## **Задание №1. Операции с массивами NumPy и геопространственными координатами**


1. Создайте двумерный массив NumPy, содержащий широту и долготу следующих городов: Токио (35.6895, 139.6917), Нью-Йорк (40.7128, -74.0060), Лондон (51.5074, -0.1278) и Париж (48.8566, 2.3522).


In [ ]:
import numpy as np

city_names = np.array(["Токио", "Нью-Йорк", "Лондон", "Париж"])
coords = np.array([
    [35.6895, 139.6917],
    [40.7128, -74.0060],
    [51.5074, -0.1278],
    [48.8566, 2.3522],
])

coords

2. Преобразуйте значения широты и долготы из градусов в радианы с помощью функции np.radians().


In [ ]:
coords_rad = np.radians(coords)

coords_rad



3. Рассчитайте поэлементную разницу между координатами Токио и других городов в радианах.

In [ ]:
tokyo_coords_rad = coords_rad[0]
diff_from_tokyo = coords_rad[1:] - tokyo_coords_rad

for city, diff in zip(city_names[1:], diff_from_tokyo):
    print(f"{city}: широта {diff[0]:.6f}, долгота {diff[1]:.6f}")

## **Задание 2. Операции с DataFrame Pandas и геопространственными данными**


1. Загрузите набор данных о городах мира по следующему URL с помощью Pandas: https://github.com/opengeos/datasets/releases/download/world/world_cities.csv




```python
import pandas as pd
# Pandas поддерживает загрузку данных по прямым ссылкам из сети Интернет
url = "https://github.com/opengeos/datasets/releases/download/world/world_cities.csv"

df = pd.read_csv(url)
```



In [ ]:
import pandas as pd

url = "https://github.com/opengeos/datasets/releases/download/world/world_cities.csv"
df = pd.read_csv(url)

df

2. Отобразите первые 5 строк и проверьте наличие отсутствующих значений.


In [ ]:
display(df.head())

missing_values = df.isna().sum()
missing_values

3. Отфильтруйте набор данных, чтобы включить только города с населением более 1 миллиона человек.


In [ ]:
cities_over_1m = df[df["population"] > 1_000_000].copy()

cities_over_1m

4. Сгруппируйте города по странам и рассчитайте общую численность населения для каждой страны.


In [ ]:
country_population = (
    df.groupby("country", as_index=False)["population"]
    .sum()
    .sort_values("population", ascending=False)
)

country_population



5. Отсортируйте города по населению в порядке убывания и отобразите первые 10 городов.

In [ ]:
top_10_cities = df.sort_values("population", ascending=False).head(10)

top_10_cities

## **Задание №3. Создание и обработка GeoDataFrames с помощью GeoPandas**


1. Загрузите набор данных о зданиях Нью-Йорка из файла GeoJSON с помощью GeoPandas: https://github.com/opengeos/datasets/releases/download/places/nyc_buildings.geojson

In [ ]:
import geopandas as gpd

buildings_url = "https://github.com/opengeos/datasets/releases/download/places/nyc_buildings.geojson"
buildings = gpd.read_file(buildings_url)

buildings

2. Создайте график контуров зданий и раскрасьте их в зависимости от высоты здания (используйте столбец `height_MS`).


In [ ]:
ax = buildings.plot(
    column="height_MS",
    legend=True,
    figsize=(10, 10),
    edgecolor="black",
    linewidth=0.1,
    cmap="viridis",
)
ax.set_axis_off()

3. Создайте интерактивную карту контуров зданий и раскрасьте их в зависимости от высоты здания (используйте столбец `height_MS`).


In [ ]:
import leafmap

buildings_map = buildings.to_crs(epsg=4326)

m = leafmap.Map(center=(40.71, -74.0), zoom=15)
m.add_data(
    buildings_map,
    column="height_MS",
    cmap="viridis",
    scheme="Quantiles",
    k=10,
    add_legend=True,
    layer_name="Высота зданий",
)
m

4. Рассчитайте среднюю высоту зданий (используйте столбец `height_MS`).


In [ ]:
mean_height = buildings["height_MS"].mean()

print(f"Средняя высота зданий: {mean_height:.2f}")

5. Выберите здания с высотой, превышающей среднюю высоту.


In [ ]:
buildings_above_mean = buildings[buildings["height_MS"] > mean_height].copy()

buildings_above_mean




6. Сохраните GeoDataFrame в новый файл GeoJSON.

In [ ]:
output_geojson = "nyc_buildings_above_mean.geojson"
buildings_above_mean.to_file(output_geojson, driver="GeoJSON")

print(f"Файл сохранён: {output_geojson}")

## **Задание №4. Применение NumPy, Pandas и GeoPandas для обработки и анализа пространственных данных**


1. Используйте Pandas для загрузки набора данных о городах мира по следующему URL: https://github.com/opengeos/datasets/releases/download/world/world_cities.csv


In [ ]:
import pandas as pd

url = "https://github.com/opengeos/datasets/releases/download/world/world_cities.csv"
cities_df = pd.read_csv(url)

cities_df

2. Отфильтруйте набор данных, чтобы включить только города с широтой между -40 и 60 (т.е. города, расположенные в Северном полушарии или вблизи экватора).


In [ ]:
filtered_cities = cities_df[cities_df["lat"].between(-40, 60)].copy()

filtered_cities

3. Создайте GeoDataFrame из отфильтрованного набора данных, преобразовав широту и долготу в геометрии.


In [ ]:
import geopandas as gpd

cities_gdf = gpd.GeoDataFrame(
    filtered_cities,
    geometry=gpd.points_from_xy(filtered_cities["lng"], filtered_cities["lat"]),
    crs="EPSG:4326",
)

cities_gdf

4. Перепроецируйте GeoDataFrame в проекцию Меркатора (EPSG:3857).


In [ ]:
cities_mercator = cities_gdf.to_crs(epsg=3857)

cities_mercator

5. Рассчитайте расстояние (в метрах) между каждым городом и Парижем.


In [ ]:
paris = gpd.GeoSeries(
    gpd.points_from_xy([2.3522], [48.8566]),
    crs="EPSG:4326",
).to_crs(epsg=3857).iloc[0]

cities_mercator["distance_to_paris_m"] = cities_mercator.geometry.distance(paris)

cities_mercator[["city", "country", "distance_to_paris_m"]].head()



6. Отобразите города на карте мира, раскрасив точки в зависимости от их расстояния до Парижа.



> 💡 **Подсказка:** Вы можете использовать интерактивную карту `leafmap.Map()` и метод m.add_data(), передав в него ваш GeoDataFrame (предварительно возвращенный в систему координат EPSG:4326).

Основные параметры данного метода:


* column — столбец с рассчитанным расстоянием до Парижа (предварительно разделите значение расстояния на 1000, чтобы легенда отображалась в километрах).
* cmap — цветовая палитра (например, "plasma", "viridis" или "YlOrRd").
* k — количество цветовых интервалов (рекомендуется 10–15 для плавности).
* scheme — алгоритм классификации данных:
  * "Quantiles" (Квантили) — делит города на группы с равным количеством точек в каждой. Это создаст максимальный цветовой контраст: даже если города расположены кучно, вы увидите разницу между ними.
   * "EqualInterval" (Равные интервалы) — делит весь диапазон расстояний на равные отрезки в километрах (например, 0-1000 км, 1000-2000 км и т.д.). Это нагляднее показывает реальный масштаб удаленности.



* установите параметр add_legend=True, чтобы на карте появилась шкала соответствия цветов и расстояний.

In [ ]:
import leafmap

cities_map_gdf = cities_mercator.to_crs(epsg=4326).copy()
cities_map_gdf["distance_to_paris_km"] = cities_map_gdf["distance_to_paris_m"] / 1000

m = leafmap.Map(center=(20, 0), zoom=2)
m.add_data(
    cities_map_gdf,
    column="distance_to_paris_km",
    cmap="plasma",
    scheme="Quantiles",
    k=12,
    add_legend=True,
    layer_name="Расстояние до Парижа, км",
)
m